In [1]:
# Python version
import sys
print(sys.version)

3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:16:45) [MSC v.1942 64 bit (AMD64)]


In [2]:
# Import key libraries and check versions
import pandas as pd
import numpy as np
import sklearn
import irapy

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("sklearn:", sklearn.__version__)
print("irapy:", irapy.__version__)

pandas: 2.3.3
numpy: 2.2.5
sklearn: 1.8.0
irapy: 0.1.2


In [3]:
# Import functions
from irapy import single_ira, repeated_ira
import matplotlib.pyplot as plt
from sklearn.datasets import make_friedman1
from sklearn.ensemble import RandomForestRegressor

In [4]:
# Number of cpus
n_cpus = 12

In [5]:
# Create a nonlinear dataset
X, y = make_friedman1(
    n_samples=5000,
    n_features=8,
    noise=6,
    random_state=42
)

In [6]:
# Convert to dataframe
X_ = pd.DataFrame(X,columns = [f"X{i}" for i in range(1,9,1)])
y_ = pd.Series(y,name = 'y')

In [7]:
# Check the descriptive statistics
df = pd.concat([X_,y_],axis=1)
df_summary = df.describe().T.reset_index(drop=False)
df_summary

,index,count,mean,std,min,25%,50%,75%,max
0,X1,5000.0,0.501656,0.284027,0.000110,0.260730,0.508340,0.735470,0.999871
1,X2,5000.0,0.492862,0.290038,0.000048,0.237528,0.490416,0.741440,0.999732
2,X3,5000.0,0.496849,0.286504,0.000310,0.249203,0.495692,0.744504,0.999925
3,X4,5000.0,0.503394,0.288367,0.000053,0.258138,0.500781,0.752098,0.999901
4,X5,5000.0,0.497566,0.287813,0.000135,0.247526,0.495818,0.746784,0.999696
5,X6,5000.0,0.500082,0.289195,0.000195,0.250692,0.501786,0.751967,0.999789
6,X7,5000.0,0.502724,0.288260,0.000006,0.252691,0.504194,0.751266,0.999411
7,X8,5000.0,0.499774,0.291296,0.000031,0.244374,0.504017,0.752889,0.999505
8,y,5000.0,14.263708,7.834022,-12.124279,8.846967,14.277739,19.542211,44.209359


In [8]:
df_summary.to_csv('nonlinear_data_summary.csv',index=False)

In [9]:
# Build a nonlinear regression model (Random Forest regressor)
rf = RandomForestRegressor(random_state=42).fit(X_, y_)
r2 = rf.score(X_, y_)
print(f"Model trained — R² score on training data: {r2:.4f}")

Model trained — R² score on training data: 0.9061


In [10]:
#  Run Single-execution IRA (No CI)

def ira_cal():
    column_name = ['N. background samp', 'N. interpl'] + list(X_.columns)
    result = []
    step = 1
    total_steps = 16
    
    for n in [50, 100, 200, 500]:
        for i in [50, 100, 200, 500]:
            
            print(f"[Single IRA] Step {step}/{total_steps}: n_background={n}, n_interpol={i}")
            
            ira_result = single_ira(X_, rf, num_interpol=i, num_background_samples=n, random_state=42)
            
            # check the result of the first one (n = 50; i = 50)
            if step==1:
                display(ira_result)
            
            result.append([n, i] + ira_result.iloc[:, 1].tolist())
            step += 1
    result_df = pd.DataFrame(result, columns=column_name)
    return result_df

single_ira_result = ira_cal()
print("\n--- Final IRA Result Table (Single execution) ---")
display(single_ira_result)

[Single IRA] Step 1/16: n_background=50, n_interpol=50


,predictor,ira_value
0,X1,7.759426
1,X2,8.877953
2,X3,5.328633
3,X4,12.045244
4,X5,6.658849
5,X6,2.504934
6,X7,2.327400
7,X8,2.432924


[Single IRA] Step 2/16: n_background=50, n_interpol=100
[Single IRA] Step 3/16: n_background=50, n_interpol=200
[Single IRA] Step 4/16: n_background=50, n_interpol=500
[Single IRA] Step 5/16: n_background=100, n_interpol=50
[Single IRA] Step 6/16: n_background=100, n_interpol=100
[Single IRA] Step 7/16: n_background=100, n_interpol=200
[Single IRA] Step 8/16: n_background=100, n_interpol=500
[Single IRA] Step 9/16: n_background=200, n_interpol=50
[Single IRA] Step 10/16: n_background=200, n_interpol=100
[Single IRA] Step 11/16: n_background=200, n_interpol=200
[Single IRA] Step 12/16: n_background=200, n_interpol=500
[Single IRA] Step 13/16: n_background=500, n_interpol=50
[Single IRA] Step 14/16: n_background=500, n_interpol=100
[Single IRA] Step 15/16: n_background=500, n_interpol=200
[Single IRA] Step 16/16: n_background=500, n_interpol=500

--- Final IRA Result Table (Single execution) ---


,N. background samp,N. interpl,X1,X2,X3,X4,X5,X6,X7,X8
0,50,50,7.759426,8.877953,5.328633,12.045244,6.658849,2.504934,2.327400,2.432924
1,50,100,7.855324,9.154776,5.407117,12.120668,6.766085,2.566316,2.383337,2.485925
2,50,200,7.955261,9.243118,5.435038,12.174663,6.789185,2.586902,2.428681,2.504361
3,50,500,8.010986,9.338264,5.453292,12.300153,6.897353,2.613914,2.445388,2.527780
4,100,50,7.759089,8.863136,5.241224,12.091714,6.467612,2.625670,2.381790,2.442522
5,100,100,7.844571,9.137447,5.317583,12.204320,6.570078,2.689004,2.432207,2.495990
6,100,200,7.942154,9.243029,5.348418,12.266740,6.631080,2.717494,2.465376,2.516863
7,100,500,8.001912,9.339822,5.368645,12.363168,6.721293,2.741272,2.486642,2.540949
8,200,50,8.048571,8.749499,5.335665,11.970136,6.575531,2.620657,2.422437,2.511285
9,200,100,8.163901,8.998135,5.407905,12.078102,6.672711,2.685738,2.479854,2.559101


In [11]:
single_ira_result.to_csv('nonlinear_example_single_ira_result.csv',index=False)

In [12]:
# Run Repeated IRA (with CI)

def run_repeated():
    df_list = [pd.Series(list(X_), name='predictor')]

    for i in [10, 30, 50, 70, 90]:
        print(f"[Repeated IRA] Running {i} repeats...")
        repeated_result = repeated_ira(
            input_data=X_,
            model=rf,
            n_repeats=i,
            random_state=42,
            n_jobs=n_cpus
        )
        
        display(repeated_result)
        repeated_result = repeated_result.iloc[:, 1:].copy()
        repeated_result = repeated_result.rename(columns={
            "mean_ira": f"mean_{i}",
            "ci_lower_ira": f"ci_lower_{i}",
            "ci_upper_ira": f"ci_upper_{i}"
        })
        df_list.append(repeated_result)

    df_result = pd.concat(df_list, axis=1)
    return df_result

repeated_ira_result = run_repeated()
print("\n--- Final Repeated IRA Result Table ---")
display(repeated_ira_result)

[Repeated IRA] Running 10 repeats...


,predictor,mean_ira,ci_lower_ira,ci_upper_ira
0,X1,8.292618,7.935530,8.778709
1,X2,9.127367,8.583681,9.506050
2,X3,5.318254,5.137296,5.587676
3,X4,11.727112,11.489481,11.977452
4,X5,6.645129,6.280097,6.940188
5,X6,2.721342,2.545855,2.906426
6,X7,2.408493,2.303542,2.519958
7,X8,2.414514,2.278944,2.551369


[Repeated IRA] Running 30 repeats...


,predictor,mean_ira,ci_lower_ira,ci_upper_ira
0,X1,8.260555,7.896231,8.692583
1,X2,9.086974,8.540903,9.520244
2,X3,5.326872,5.040029,5.590241
3,X4,11.689859,11.296412,11.973086
4,X5,6.652838,6.237317,6.983167
5,X6,2.728049,2.577713,2.903340
6,X7,2.404733,2.297412,2.549932
7,X8,2.429516,2.295268,2.561628


[Repeated IRA] Running 50 repeats...


,predictor,mean_ira,ci_lower_ira,ci_upper_ira
0,X1,8.267965,7.827019,8.627061
1,X2,9.168145,8.606849,9.572007
2,X3,5.337581,5.051481,5.702426
3,X4,11.667085,11.284521,11.977452
4,X5,6.700893,6.275655,7.065130
5,X6,2.735810,2.595121,2.891700
6,X7,2.416347,2.283107,2.558200
7,X8,2.437180,2.308917,2.562245


... (2 outputs left out)

[Repeated IRA] Running 90 repeats...


,predictor,mean_ira,ci_lower_ira,ci_upper_ira
0,X1,8.279939,7.842071,8.801345
1,X2,9.146681,8.519357,9.573335
2,X3,5.349097,5.051899,5.681818
3,X4,11.632612,11.246770,11.981651
4,X5,6.707672,6.362897,7.056514
5,X6,2.705788,2.533889,2.886098
6,X7,2.423446,2.283107,2.558200
7,X8,2.445799,2.293875,2.600172



--- Final Repeated IRA Result Table ---


,predictor,mean_10,ci_lower_10,ci_upper_10,mean_30,ci_lower_30,ci_upper_30,mean_50,ci_lower_50,ci_upper_50,mean_70,ci_lower_70,ci_upper_70,mean_90,ci_lower_90,ci_upper_90
0,X1,8.292618,7.935530,8.778709,8.260555,7.896231,8.692583,8.267965,7.827019,8.627061,8.281446,7.849129,8.836515,8.279939,7.842071,8.801345
1,X2,9.127367,8.583681,9.506050,9.086974,8.540903,9.520244,9.168145,8.606849,9.572007,9.181906,8.621470,9.611636,9.146681,8.519357,9.573335
2,X3,5.318254,5.137296,5.587676,5.326872,5.040029,5.590241,5.337581,5.051481,5.702426,5.347583,5.056433,5.698415,5.349097,5.051899,5.681818
3,X4,11.727112,11.489481,11.977452,11.689859,11.296412,11.973086,11.667085,11.284521,11.977452,11.639137,11.234070,11.973086,11.632612,11.246770,11.981651
4,X5,6.645129,6.280097,6.940188,6.652838,6.237317,6.983167,6.700893,6.275655,7.065130,6.713691,6.329890,7.064762,6.707672,6.362897,7.056514
5,X6,2.721342,2.545855,2.906426,2.728049,2.577713,2.903340,2.735810,2.595121,2.891700,2.721443,2.547988,2.891251,2.705788,2.533889,2.886098
6,X7,2.408493,2.303542,2.519958,2.404733,2.297412,2.549932,2.416347,2.283107,2.558200,2.416036,2.275422,2.568849,2.423446,2.283107,2.558200
7,X8,2.414514,2.278944,2.551369,2.429516,2.295268,2.561628,2.437180,2.308917,2.562245,2.439372,2.301778,2.569512,2.445799,2.293875,2.600172


In [13]:
repeated_ira_result.to_csv('nonlinear_example_repeated_ira_result.csv',index=False)